# Test python verification packages with real dataset

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
print("Python version")
print (sys.version)
#print("Version info.")
#print (sys.version_info)

Python version
3.10.11 | packaged by conda-forge | (main, May 10 2023, 18:58:44) [GCC 11.3.0]


In [3]:
import numpy as np
import pandas as pd
import xarray as xr
import scoringrules
from scores.probability import crps_for_ensemble
import properscoring
sys.path.append('/datasets/work/lw-hydrofct/work/common/Software/python_verification/')
import vrf_scores
import pymvscore

In [14]:
## Load dataset
streamflow = xr.open_dataset('stremflow_data_for_scores.nc')
streamflow

<xarray.Dataset> Size: 12MB
Dimensions:     (station: 16, time: 3743, ens_member: 50)
Coordinates:
  * station     (station) int32 64B 104 607 207 511 515 ... 609 517 603 605 514
  * time        (time) datetime64[ns] 30kB 2012-04-05T23:00:00 ... 2022-07-04...
  * ens_member  (ens_member) int32 200B 1 2 3 4 5 6 7 8 ... 44 45 46 47 48 49 50
Data variables:
    obs         (time, station) float32 240kB ...
    fcst        (time, ens_member, station) float32 12MB ...

## Run scores

In [21]:
from numba import jit
@jit(nopython=True)
def crps_from_empirical_cdf(pred, obs, dim=0):
    #https://docs.nvidia.com/deeplearning/modulus/modulus-core/_modules/modulus/metrics/general/crps.html
    n = pred.shape[dim]
    pred = np.sort(pred, axis=dim)
    ans = np.zeros_like(obs)

    # dx [F(x) - H(x-y)]^2 = dx [0 - 1]^2 = dx
    # val = ensemble[0] - truth
    val = (pred[0, :] - obs)
    #val = (pred[:, 0] - obs)
    ans += np.maximum(val, 0.0)

    for i in range(n - 1):
        x0 = pred[i, :]
        x1 = pred[i+1, :]

        cdf = (i + 1) / n

        # a. case y < x0
        val = (x1 - x0) * (cdf - 1) ** 2
        mask = obs < x0
        ans += val * mask

        # b. case x0 <= y <= x1
        val = (obs - x0) * cdf**2 + (x1 - obs) * (cdf - 1) ** 2
        mask = (obs >= x0) & (obs <= x1)
        ans += val * mask

        # c. case x1 < t
        mask = obs > x1
        val = (x1 - x0) * cdf**2
        ans += val * mask

    # dx [F(x) - H(x-y)]^2 = dx [1 - 0]^2 = dx
    val = obs - pred[-1, :]
    ans += np.maximum(val, 0.0)
    return ans

In [8]:
def energy_score(forecasts, obs):

    # must have dimensions of (num_variables, num_ensembles). Variables can include different locations, time steps, climate variables etc

    num_samples = forecasts.shape[1]

    s1 = np.sqrt(np.sum(np.square(forecasts - obs[:, np.newaxis]), axis=0)).sum()

    pairwise_diffs = forecasts[:, :, np.newaxis] - forecasts[:, np.newaxis, :]
    s2 = np.sqrt(np.sum(np.square(pairwise_diffs), axis=0)).sum()

    es = (s1 / num_samples) - s2 / (2 * num_samples**2)
    
    return es

In [8]:
def energy_score_multiple(f_ens, o,return_mean=True):
    
    #num_samples = f_ens.shape[1]
    shape = f_ens.shape
    # handle nans
    mean_fcst = np.mean(f_ens,axis=1) #update by Durga
    nanindx = ~np.isnan(o) & ~np.isnan(mean_fcst)    
    #print(nanindx)
    o = o[nanindx]
    f_ens = f_ens[nanindx,:]    
    if len(shape) == 2 and shape[0] > 1 and shape[1] > 1:  # check for matrix
        num_events = f_ens.shape[0]
        num_ens = f_ens.shape[1]
        crps = []
        for i in range(num_events):
            #ff = f_ens[i,:].reshape(1,num_ens)
            ff = f_ens[i,:]
            ff = ff[np.newaxis, :]   # should be size of 1 by num_samples  
            oo = o[i]
            oo = np.array([oo])
            crps.append(energy_score(ff, oo))   # obs should be 
        #print(f"Fcst shape {ff.shape}, Obs shape: {oo.shape}")
        if return_mean:
            return np.mean(crps)
        else:
            return np.asarray(crps)
    else: 
        return energy_score(f_ens, o)  

In [15]:
def remove_nans(fcst,obs,site):
    ff = fcst.isel(station=site).values
    oo = obs.isel(station=site).values   
    mean_fcst = np.mean(ff,axis=1) 
    nanindx = ~np.isnan(oo) & ~np.isnan(mean_fcst)
    oo = oo[nanindx]
    ff = ff[nanindx,:] 
    return ff,oo

In [18]:
nsite = streamflow['fcst'].shape[2]
nsite

16

In [20]:
streamflow

<xarray.Dataset> Size: 12MB
Dimensions:     (station: 16, time: 3743, ens_member: 50)
Coordinates:
  * station     (station) int32 64B 104 607 207 511 515 ... 609 517 603 605 514
  * time        (time) datetime64[ns] 30kB 2012-04-05T23:00:00 ... 2022-07-04...
  * ens_member  (ens_member) int32 200B 1 2 3 4 5 6 7 8 ... 44 45 46 47 48 49 50
Data variables:
    obs         (time, station) float32 240kB ...
    fcst        (time, ens_member, station) float32 12MB 0.009 1.8 ... nan nan

### 1. scores.probability.crps_for_ensemble

In [11]:
%%time
# Specifying method='ecdf' assumes the empirical CDF
scores_crps = crps_for_ensemble(streamflow['fcst'], streamflow['obs'], ensemble_member_dim='ens_member', method='ecdf',preserve_dims='station')

CPU times: user 514 ms, sys: 340 ms, total: 854 ms
Wall time: 1.09 s


### 2. scoringrules.crps_ensemble

In [12]:
%%time
ens_mem_dim = streamflow['fcst'].get_axis_num("ens_member")
scoringrules_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    scoringrules_crps_vals = scoringrules.crps_ensemble(obs,fcst, axis=1, estimator="nrg")
    scoringrules_crps.append(scoringrules_crps_vals.mean().item())

CPU times: user 179 ms, sys: 67.7 ms, total: 246 ms
Wall time: 246 ms


### 3. properscoring.crps_ensemble

In [13]:
%%time
properscoring_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    properscoring_crps.append(np.mean(properscoring.crps_ensemble(obs,fcst)) )    

CPU times: user 63.5 ms, sys: 203 µs, total: 63.7 ms
Wall time: 62.9 ms


### 4. vrf_scores.crps_ecdf_multiple

In [14]:
%%time
vrf_scores_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    vrf_scores_crps.append(vrf_scores.crps_ecdf_multiple(fcst,obs)) 

CPU times: user 4.52 s, sys: 25.3 ms, total: 4.54 s
Wall time: 4.53 s


In [15]:
vrf_scores_crps

[0.6658845852402648,
 5.458856099202116,
 1.1149853560535183,
 0.5313843083307749,
 3.184141781398476,
 2.3757416331301626,
 2.8585770427529407,
 13.331986247911903,
 36.138535156203844,
 35.97743905921267,
 47.7122135541139,
 38.966512165256944,
 23.099913879862477,
 415.9833152146851,
 40.12662869157369,
 31.135090439326294]

In [17]:
fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],11)
vrf_scores.crps_ecdf_multiple(fcst,obs)

38.966512165256944

In [18]:
fcst.shape

(3677, 50)

In [19]:
obs.shape

(3677,)

### 5. crps_from_empirical_cdf

In [24]:
%%time
ecdf_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    ecdf_crps.append(np.mean(crps_from_empirical_cdf(fcst.T,obs,dim=0))) 

CPU times: user 90.1 ms, sys: 116 µs, total: 90.2 ms
Wall time: 89.5 ms


### 6. energy score

In [25]:
%%time
eng_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    eng_crps.append(energy_score_multiple(fcst,obs)) 

CPU times: user 1.22 s, sys: 0 ns, total: 1.22 s
Wall time: 1.22 s


### make dataframe

In [27]:
df = pd.DataFrame(scores_crps.values,columns = ['scores.crps(871 ms)'])
df['scoringrules_crps(264 ms)'] = scoringrules_crps
df['properscoring.crps_ensemble(63.1 ms)'] = properscoring_crps
df['vrf_scores_crps(4.58 s)']  = vrf_scores_crps
df['ecdf_crps(89.5 ms)'] = ecdf_crps
df['eng_crps(1.22 s)'] = eng_crps
df.index.name = 'sites'
df

,scores.crps(871 ms),scoringrules_crps(264 ms),properscoring.crps_ensemble(63.1 ms),vrf_scores_crps(4.58 s),ecdf_crps(89.5 ms),eng_crps(1.22 s)
sites,,,,,,
0,0.665885,0.665885,0.665885,0.665885,0.665885,0.665885
1,5.458856,5.458856,5.458856,5.458856,5.458857,5.458856
2,1.114985,1.114985,1.114985,1.114985,1.114985,1.114985
3,0.531385,0.531384,0.531384,0.531384,0.531384,0.531384
4,3.184142,3.184142,3.184142,3.184142,3.184142,3.184142
5,2.375742,2.375742,2.375742,2.375742,2.375742,2.375742
6,2.858582,2.858577,2.858577,2.858577,2.858577,2.858577
7,13.331986,13.331985,13.331986,13.331986,13.331985,13.331986
8,36.138537,36.138538,36.138535,36.138535,36.138535,36.138535


## Energy_Score

In [11]:
num_ens = 500 
ens1, obs1 = np.random.normal(200,20,num_ens), 150.
ens2, obs2 = np.random.normal(150,30,num_ens), 170.
forecasts = [ens1, ens2]
obs = [obs1, obs2]

In [9]:
%%timeit
#crps_scores = [crps_from_empirical_cdf(e, o) for e,o in zip(forecasts,obs)]
energy_scores = [energy_score(np.array([e]), np.array([o])) for e,o in zip(forecasts,obs)]
for e,o in zip(forecasts,obs):
    energy_scores1 = energy_score(np.array([e]), np.array([o]))
#print('crps_ecdf:       ', crps_scores, '  mean:', np.mean(crps_scores))
#print('optimised energy:', energy_scores, '  mean:', np.mean(energy_scores))

6.29 ms ± 32.4 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## MVScore

In [10]:
%%timeit
#num_ens = 500
#ens1, obs1 = np.random.normal(200,20,num_ens), 150.
#ens2, obs2 = np.random.normal(150,30,num_ens), 170.
#forecasts = [ens1, ens2]
#obs = [obs1, obs2] 
ms = pymvscore.PyMultivariateScore()
score_cpp = ms.crpsECDF_many(forecasts, obs)

46.5 µs ± 391 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [5]:
score_cpp

24.604201519426898

In [12]:
forecasts

[array([192.8057262 , 211.35051068, 209.8577137 , 228.71174287,
        199.98205134, 191.523498  , 226.62830555, 164.1282447 ,
        166.27768826, 178.52102982, 238.55930851, 223.86601436,
        157.47928085, 209.70682509, 219.07156616, 183.99653158,
        203.96044318, 171.84390297, 185.31328432, 226.66109381,
        182.41599123, 207.55757026, 191.83515413, 191.76028357,
        196.62440904, 172.27372899, 176.10709554, 198.04189423,
        187.73707988, 171.10017782, 191.71797269, 172.87194775,
        195.00067756, 186.3923851 , 229.24375754, 182.07065055,
        185.52673302, 195.59927767, 173.18802501, 191.22910642,
        212.83243645, 221.99706196, 231.44188666, 185.27754072,
        210.09092174, 243.32626854, 215.19649168, 198.8118459 ,
        211.93087061, 203.41250206, 229.23618504, 196.00831262,
        189.67045459, 182.84244095, 185.3262983 , 210.75036841,
        213.24431918, 241.36509551, 227.11608469, 211.15960109,
        220.56805354, 179.67110813, 182.

In [20]:
def mvscore_multiple(f_ens, o,return_mean=True):
    ms = pymvscore.PyMultivariateScore()
    #num_samples = f_ens.shape[1]
    shape = f_ens.shape
    # handle nans
    mean_fcst = np.mean(f_ens,axis=1) #update by Durga
    nanindx = ~np.isnan(o) & ~np.isnan(mean_fcst)    
    #print(nanindx)
    o = o[nanindx]
    f_ens = f_ens[nanindx,:]    
    if len(shape) == 2 and shape[0] > 1 and shape[1] > 1:  # check for matrix
        num_events = f_ens.shape[0]
        num_ens = f_ens.shape[1]
        crps = []
        for i in range(num_events):
            #ff = f_ens[i,:].reshape(1,num_ens)
            ff = f_ens[i,:]
            ff = ff[np.newaxis, :]   # should be size of 1 by num_samples  
            oo = o[i]
            oo = np.array([oo])
            #crps.append(energy_score(ff, oo))   # obs should be 
            crps.append(ms.crpsECDF_many(ff, oo))
        #print(f"Fcst shape {ff.shape}, Obs shape: {oo.shape}")
        if return_mean:
            return np.mean(crps)
        else:
            return np.asarray(crps)
    else: 
        return ms.crpsECDF_many(f_ens, o)  

In [21]:
%%time
mvscore_crps = []
for s in range(nsite):
    fcst,obs = remove_nans(streamflow['fcst'],streamflow['obs'],s)
    mvscore_crps .append(mvscore_multiple(fcst,obs)) 

CPU times: user 349 ms, sys: 12.2 ms, total: 361 ms
Wall time: 360 ms


In [22]:
mvscore_crps

[0.6658845880100632,
 5.458856096315842,
 1.1149853531849896,
 0.5313843076093614,
 3.1841417898583764,
 2.375741629164669,
 2.858577051263099,
 13.331986217228263,
 36.13853519210955,
 35.97743880576042,
 47.7122136804166,
 38.96651209180809,
 23.099913891183093,
 415.98331471624806,
 40.126628691573686,
 31.135090645592086]